# RAIL lung-nodule counting — Colab setup

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive. This pipeline only ever uses patients 1-20 (every script defaults to `--start 1 --end 20`), so you don't need the full 99-patient/11GB `LDIC-IDRI-subset` — just those 20 patients + `annotations.csv`, zipped to ~1.1GB. (If you need more patients later, re-zip a wider range the same way.)
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [4]:
!git clone https://github.com/freya-gul/rail.git
%cd rail

Cloning into 'rail'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 83 (delta 22), reused 77 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 74.20 MiB | 19.67 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Updating files: 100% (49/49), done.
Encountered 2 files that should have been pointers, but weren't:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
/content/rail


Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path every script's `DICOM_ROOT` already expects, so nothing else needs to change:

In [5]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [6]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

annotations.csv  lidc_idri


In [7]:
# This notebook only runs the MedGemma counting baseline (8_evaluate_medgemma_counting.py),
# which doesn't touch pylidc/monai/simpleitk at all - only pydicom + transformers are needed
# beyond what Colab already has. See requirements-colab.txt for the full install if you also
# want to run the MONAI detector (7_evaluate_detection.py) or regenerate ground truth (6_*.py).
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.9 MB/s eta 0:00:00


Sanity check: GPU visible to torch, and the detector/MedGemma scripts will now pick it (they already default to `cuda` > `mps` > `cpu`):

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA A100-SXM4-40GB


## Run MedGemma counting

Compares MedGemma 1.5's whole-volume nodule count against two independent ground truths, already committed in the repo for patients 1-20:
- `gt_count_pylidc` — every pylidc consensus annotation cluster (`ground_truth_annotations.json`)
- `gt_count_luna16` — the actual external LUNA16 challenge `annotations.csv`, joined by SeriesInstanceUID (a separately-collected nodule list, not derived from pylidc)

In [9]:
!git pull

Already up to date.


In [ ]:
# MedGemma-only whole-volume counting baseline vs. ground truth (GPU)
!python image_download/8_evaluate_medgemma_counting.py --start 1 --end 20

Loading MedGemma 1.5 on cuda...
config.json: 100% 2.55k/2.55k [00:00<00:00, 7.04MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 63.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.96G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  67% 5.77G/8.60G [00:16<00:07, 386MB/s,  149MB/s  ]
Reconstructing (incomplete total...):  74% 6.37G/8.60G [00:17<00:05, 420MB/s,  145MB/s  ]
Reconstructing (incomplete total...):  80% 6.86G/8.60G [00:20<00:06, 275MB/s,  161MB/s  ]
Reconstructing (incomplete total...):  94% 8.07G/8.60G [00:20<00:01, 529MB/s,  162MB/s  ]
Reconstructing (incomplete total...): 100% 8.60G/8.60G [00:21<00:00, 503MB/s,  208MB/s  ]

Fetching 2 files: 100% 2/2 [00:21<00:00, 10.97s/it]
Download complete: 100% 7.79G/7.79G [00:21<00:00, 781MB/s,  175MB/s  ]
Download complete: 100% 7.79G/7.79G [00:22<00:00, 354MB/s,  175MB/

Results land at `image_download/medgemma_counting_comparison.csv`, with per-patient `predicted_count`, `gt_count_pylidc`, `gt_count_luna16`, and both error columns, plus MAE/bias vs. each ground truth printed above. The run caches each patient's MedGemma response to `medgemma_count_cache/<patient>.json`, so it's safe to interrupt and rerun — already-cached patients are skipped.